# 🏷️ Data Types & Categorical Data in Pandas

---

## Why Data Types Matter

In pandas, every column has a **dtype** (data type). The dtype determines:
- How much **memory** the column uses
- What **operations** are valid on it (e.g., you can't sum a text column)
- How pandas **displays** and **groups** the data
- Whether a column can be used as a **categorical** (factor) variable

A common source of bugs in data analysis:
> "Why is my `total_bill` column returning wrong averages?"  
> Answer: It was stored as `object` (string), not `float64`.

Understanding and controlling dtypes is a core data-cleaning skill.

---

## What You Will Learn

| Section | Topics |
|---|---|
| **Part 1: Setup** | Loading data, checking dtypes |
| **Part 2: Converting to Strings** | `astype(str)`, when object ≠ string |
| **Part 3: Converting to Numerics** | `astype(float/int)`, `pd.to_numeric()`, handling errors |
| **Part 4: Categorical Data** | What categories are, memory benefits, `astype('category')` |
| **Part 5: Working with Categories** | `.cat` accessor, codes, ordering, adding/removing categories |

---

## Pandas Dtype Reference

| Pandas Dtype | Meaning | Python Equivalent | Example Values |
|---|---|---|---|
| `int64` | Integer | `int` | `1`, `42`, `-7` |
| `float64` | Decimal number | `float` | `3.14`, `0.0`, `-2.5` |
| `object` | Text (or mixed) | `str` | `'Male'`, `'Sunday'` |
| `bool` | True/False | `bool` | `True`, `False` |
| `datetime64` | Date and time | — | `2024-06-15` |
| `category` | Fixed set of labels | — | `'Male'/'Female'` |
| `int8`, `int16`, `int32` | Smaller integers | — | Memory-efficient variants |

---

# Part 1: Setup — Load and Inspect the Data

---

We use the **Tips dataset** — a classic sample dataset recording restaurant bills, tips, and diner information (244 rows).

**Columns:**

| Column | Description | Expected dtype |
|---|---|---|
| `ID` | Row identifier | `int64` |
| `total_bill` | Total bill amount (£) | `float64` |
| `tip` | Tip amount (£) | `float64` |
| `gender` | Diner gender | `object` (should be category) |
| `smoker` | Smoker or not | `object` (should be category) |
| `day` | Day of the week | `object` (should be category) |
| `time` | Lunch or Dinner | `object` (should be category) |
| `size` | Party size | `int64` |

In [ ]:
# Suppress deprecation and future warnings to keep output clean
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Import pandas — the primary library for tabular data manipulation
import pandas as pd

# Import numpy for NaN handling
import numpy as np

# Import StringIO so we can build a CSV in memory (makes notebook self-contained)
# In real usage: df_tips = pd.read_csv('path/to/tips.csv')
from io import StringIO

# Construct the tips dataset as a CSV string
# This is equivalent to having a tips.csv file on disk
tips_csv = """ID,total_bill,tip,gender,smoker,day,time,size
1,16.99,1.01,Female,No,Sun,Dinner,2
2,10.34,1.66,Male,No,Sun,Dinner,3
3,21.01,3.50,Male,No,Sun,Dinner,3
4,23.68,3.31,Male,No,Sun,Dinner,2
5,24.59,3.61,Female,No,Sun,Dinner,4
6,25.29,4.71,Male,No,Sun,Dinner,4
7,8.77,2.00,Male,No,Sun,Dinner,2
8,26.88,3.12,Male,No,Sun,Dinner,4
9,15.04,1.96,Male,No,Sun,Dinner,2
10,14.78,3.23,Male,No,Sun,Dinner,2
11,10.27,1.71,Male,No,Sun,Dinner,2
12,35.26,5.00,Female,No,Sun,Dinner,4
13,15.42,1.57,Male,No,Sun,Dinner,2
14,18.43,3.00,Male,No,Sun,Dinner,4
15,14.83,3.02,Female,No,Sun,Dinner,2
16,21.58,3.92,Male,No,Sun,Dinner,2
17,10.33,1.67,Female,No,Sun,Dinner,3
18,16.29,3.71,Male,No,Sun,Dinner,3
19,16.97,3.50,Female,No,Sun,Dinner,3
20,20.65,3.35,Male,No,Sat,Dinner,3"""

# Read the CSV string into a DataFrame using StringIO to simulate a file object
df_tips = pd.read_csv(StringIO(tips_csv))

print("=== Dataset shape ===")
print(f"Rows: {df_tips.shape[0]}, Columns: {df_tips.shape[1]}")

print("\n=== First 5 rows ===")
print(df_tips.head())

print("\n=== Data types of each column ===")
# .dtypes returns a Series where the index is column names and values are their dtypes
print(df_tips.dtypes)

In [ ]:
# .info() gives a comprehensive summary: dtypes, non-null counts, and memory usage
# This is the first thing you should run when exploring a new dataset
print(df_tips.info())

print()
# Memory usage — important for large datasets
print("=== Memory usage per column ===")
# deep=True: accurately measure actual memory including string content
print(df_tips.memory_usage(deep=True))
print(f"\nTotal: {df_tips.memory_usage(deep=True).sum():,} bytes")

---

# Part 2: Converting to Strings

---

## 2.1 The `astype()` Function

`astype()` is the primary method for converting a Series or DataFrame column from one dtype to another.

**Syntax:**
```python
series.astype(dtype)
```

The `dtype` parameter accepts:
- **Python built-in types**: `str`, `int`, `float`, `bool`, `complex`
- **Pandas types**: `'category'`, `'datetime64[ns]'`
- **NumPy types**: `np.int32`, `np.float32`, `np.int8`
- **String abbreviations**: `'int64'`, `'float32'`, `'object'`

---

## 2.2 `object` vs `string` — An Important Distinction

In pandas, text columns appear as dtype `object`, not `string`.

| Dtype | What it means | Memory layout |
|---|---|---|
| `object` | Python object (usually str, but could be mixed) | Flexible, but slow |
| `string` (pandas 1.0+) | Explicitly string type, with better NA handling | Uses `pd.NA` not `NaN` |

When you call `astype(str)`, the column dtype shows as `object` — this is normal. The underlying values are Python strings.

In [ ]:
print("=== BEFORE conversion ===")
# Check the dtype of the 'gender' column before any conversion
# It reads as 'object' because pandas stores text as Python objects
print(f"'gender' dtype before: {df_tips['gender'].dtype}")

print()
print("=== Converting 'gender' to explicit string type ===")
# astype(str): explicitly convert the column to string type
# Assigns the result to a NEW column 'gender_str' (original column preserved)
df_tips['gender_str'] = df_tips['gender'].astype(dtype=str)

# The resulting dtype shows as 'object' — this is correct for pandas string columns
print(f"'gender_str' dtype after:  {df_tips['gender_str'].dtype}")

print()
print("Both show 'object' — but the conversion ensures all values are Python str objects")
print()

# Confirm the values are identical after conversion
print("=== First 5 values comparison ===")
comparison = pd.DataFrame({
    'gender (original)': df_tips['gender'].head(),
    'gender_str (converted)': df_tips['gender_str'].head()
})
print(comparison.to_string())

In [ ]:
# --- Why would you convert to string? ---
# Common use cases for astype(str):

print("=== USE CASE 1: Convert numeric ID to string for text operations ===")
# IDs are often stored as int64 but need to be treated as strings
# e.g., to pad with zeros, concatenate with text, or use as a label
df_tips['ID_str'] = df_tips['ID'].astype(str)

# Now we can do string operations — zero-pad to 4 digits
df_tips['ID_padded'] = df_tips['ID_str'].str.zfill(4)
print(df_tips[['ID', 'ID_str', 'ID_padded']].head())

print()
print("=== USE CASE 2: Combine two columns into a single label ===")
# Convert size (int) to string so we can concatenate with text
df_tips['party_label'] = 'Party of ' + df_tips['size'].astype(str)
print(df_tips[['size', 'party_label']].head())

# Clean up the temporary columns we added for this example
df_tips.drop(columns=['ID_str', 'ID_padded', 'party_label'], inplace=True)

---

# Part 3: Converting to Numeric Values

---

## 3.1 Using `astype()` for Clean Numeric Conversion

When a column contains clean numeric data stored as strings (common after reading CSVs with formatting issues), `astype(float)` or `astype(int)` will convert it directly.

⚠️ **Important**: `astype()` will **raise an error** if any value cannot be converted (e.g., if a cell contains the string `'missing'`). For messy data, use `pd.to_numeric()` instead (see 3.2).

In [ ]:
print("=== STEP 1: Check current dtypes ===")
# Print all column dtypes before any conversion
print(df_tips.dtypes)

print()
print("=== STEP 2: Convert 'total_bill' from float64 → string (object) ===")
# astype(str): convert the numeric total_bill column to string
# This simulates what happens when you read a CSV where numbers are stored as text
df_tips['total_bill'] = df_tips['total_bill'].astype(str)
print(f"total_bill dtype after → str: {df_tips['total_bill'].dtype}")
print(df_tips['total_bill'].head())

print()
print("=== STEP 3: Convert 'total_bill' back from string → float ===")
# astype(float): convert string representation back to float
# This works cleanly because all values are valid number strings ('16.99', '10.34', etc.)
df_tips['total_bill'] = df_tips['total_bill'].astype(float)
print(f"total_bill dtype after → float: {df_tips['total_bill'].dtype}")
print(df_tips['total_bill'].head())

print()
print("=== STEP 4: Verify all dtypes are back to original ===")
print(df_tips.dtypes)

---

## 3.2 Using `pd.to_numeric()` for Messy Data

Real-world data is messy. Columns that should be numeric often contain non-numeric entries like:
- `'missing'`, `'N/A'`, `'unknown'`, `'n/a'`
- Formatting characters: `'$1,234.56'`, `'16.99%'`
- Mixed text/numbers

Using `astype(float)` on such data **raises a `ValueError`** and crashes your script.

`pd.to_numeric()` solves this with the `errors` parameter:

| `errors` value | Behaviour | When to use |
|---|---|---|
| `'raise'` (default) | Raises `ValueError` on first invalid value | Data should be clean; you want to catch errors |
| `'coerce'` | Invalid values become `NaN` | Real-world messy data; continue despite bad values |

*(Note: `errors='ignore'` was removed in pandas 2.0 — use `'coerce'` instead)*

In [ ]:
# === Simulate a messy real-world dataset ===
# Take the first 6 rows as a small working copy
df_tmp = df_tips.head(6).copy()

# Convert total_bill to string first so we can inject non-numeric values
# In real data this would come from a CSV where some cells have text
df_tmp['total_bill'] = df_tmp['total_bill'].astype(str)

# Inject the string 'missing' into rows 1, 3, 5 to simulate data entry errors
df_tmp.loc[[1, 3, 5], 'total_bill'] = 'missing'

print("=== Messy dataset (some bills are the string 'missing') ===")
print(df_tmp[['ID', 'total_bill']].to_string())
print(f"\ndtype of total_bill: {df_tmp['total_bill'].dtype}")

print()
print("=== What happens if we try astype(float) on messy data? ===")
# This would raise a ValueError — uncomment to see the error
# df_tmp['total_bill'].astype(float)  # → ValueError: could not convert 'missing' to float
print("Commented out — would raise: ValueError: could not convert string to float: 'missing'")

In [ ]:
print("=== pd.to_numeric() with errors='raise' (default) ===")
# errors='raise': the default — crashes on the first non-numeric value
# This is useful in data pipelines where bad data should stop execution
try:
    # Wrap in try/except to demonstrate the error without crashing the notebook
    result = pd.to_numeric(df_tmp['total_bill'], errors='raise')
except ValueError as e:
    # Catch and display the error message
    print(f"Error raised (as expected): {e}")

print()
print("=" * 55)
print("pd.to_numeric() with errors='coerce'")
print("=" * 55)
# errors='coerce': silently converts any non-numeric value to NaN
# 'coerce' = 'force' — force conversion, replacing failures with NaN
# This is the most practical choice for messy real-world data
result_coerce = pd.to_numeric(df_tmp['total_bill'], errors='coerce')

print("Result with errors='coerce':")
print(result_coerce)
print(f"\nDtype after coerce: {result_coerce.dtype}")
print("Notice: 'missing' rows (1, 3, 5) became NaN — valid numbers were preserved")

print()
# After coercion, we can now use .fillna() to handle the NaN values
# For example: fill missing bills with the median bill value
median_bill = result_coerce.median()
result_filled = result_coerce.fillna(median_bill)
print(f"After filling NaN with median ({median_bill:.2f}):")
print(result_filled)

In [ ]:
print("=== Converting to specific numeric types ===")

# Convert 'size' (party size) to various integer types
# Smaller integer types use less memory — useful for large datasets

print(f"'size' as int64:  {df_tips['size'].astype('int64').dtype},  "
      f"memory: {df_tips['size'].astype('int64').memory_usage()} bytes")

# int8 can hold values from -128 to 127 — plenty for party sizes 1-6
print(f"'size' as int8:   {df_tips['size'].astype('int8').dtype},   "
      f"memory: {df_tips['size'].astype('int8').memory_usage()} bytes")

print()
print("=== Downcast automatically with to_numeric ===")
# downcast='integer': automatically uses the smallest integer type that fits
# Pandas will choose int8, int16, int32, or int64 as appropriate
result_downcast = pd.to_numeric(df_tips['size'], downcast='integer')
print(f"'size' after downcast: dtype={result_downcast.dtype}")
print("(Pandas chose the smallest integer type that fits the data range)")

---

# Part 4: Categorical Data

---

## 4.1 What is Categorical Data?

**Categorical data** is data that takes values from a **fixed, finite set of possible labels**. Instead of being a continuous number, each value belongs to one of several named groups.

**Real-world examples:**

| Column | Categories | Domain |
|---|---|---|
| `gender` | Male, Female | Restaurant tips |
| `risk_level` | High, Medium, Low | Finance / Credit |
| `asset_class` | Equity, Fixed Income, Commodity | Investment |
| `day` | Mon, Tue, Wed, Thu, Fri, Sat, Sun | General |
| `loan_status` | Approved, Pending, Rejected | Banking |
| `country` | India, US, UK, Singapore, ... | Demographics |

---

## 4.2 Why Use the `category` Dtype?

When a column is stored as `object` (string), pandas stores the **full string value for every single row**.

When stored as `category`, pandas:
1. Stores the **unique labels ONCE** in a lookup table (called the 'categories')
2. Stores a small **integer code** (0, 1, 2...) per row pointing to the lookup table

```
object dtype (100 rows of 'Male'/'Female'):
  Row 0: 'Female'  (stores 6 characters)
  Row 1: 'Male'    (stores 4 characters)
  ...
  Row 99: 'Male'   (stores 4 characters)
  → 100 full string copies

category dtype:
  Categories lookup: {0: 'Female', 1: 'Male'}  (stored ONCE)
  Row 0: 0   (1 byte)
  Row 1: 1   (1 byte)
  ...
  Row 99: 1  (1 byte)
  → 100 integers + 2 strings = much smaller!
```

**Benefits of `category` dtype:**
- 🔽 **Lower memory usage** — dramatic for high-cardinality string columns in large datasets
- ⚡ **Faster groupby operations** — grouping by integer codes is faster than string comparison
- 📊 **Ordered categories** — can define a meaningful order (Low < Medium < High)
- 🛡️ **Data validation** — only values in the defined categories are valid

In [ ]:
print("=== STEP 1: Show current 'gender' as string (object) ===")
# First ensure gender is stored as a plain string (object)
# astype('str'): convert to string — the starting point before making it categorical
df_tips['gender'] = df_tips['gender'].astype('str')

# .info() shows detailed column information including dtype and memory
print(df_tips[['gender']].info())
print(f"\nMemory used by 'gender' as object:   "
      f"{df_tips['gender'].memory_usage(deep=True)} bytes")

In [ ]:
print("=== STEP 2: Convert 'gender' to category dtype ===")
# astype('category'): converts the column from object (string) to category
# Pandas will automatically discover all unique values and create the category lookup
df_tips['gender'] = df_tips['gender'].astype('category')

# .info() now shows 'category' instead of 'object' for the gender column
print(df_tips[['gender']].info())
print(f"\nMemory used by 'gender' as category: "
      f"{df_tips['gender'].memory_usage(deep=True)} bytes")

print()
print("Note: 'category' dtype uses less memory than 'object' for columns with few unique values")

---

# Part 5: Working with Categorical Data — The `.cat` Accessor

---

## 5.1 The `.cat` Accessor

Once a column is dtype `category`, pandas gives you access to a special namespace called **`.cat`** — analogous to `.str` for strings or `.dt` for datetimes.

| `.cat` attribute / method | What it does |
|---|---|
| `.cat.categories` | The Index of unique category labels |
| `.cat.codes` | Integer code for each row (0, 1, 2...) |
| `.cat.ordered` | Whether the categories have a defined order |
| `.cat.add_categories(new)` | Add new category labels |
| `.cat.remove_categories(old)` | Remove categories (values become NaN) |
| `.cat.rename_categories(mapping)` | Rename the labels |
| `.cat.set_categories(new_list)` | Replace the entire category list |
| `.cat.reorder_categories(order)` | Change the order of categories |
| `.cat.as_ordered()` | Make unordered categories ordered |
| `.cat.as_unordered()` | Remove ordering from ordered categories |

In [ ]:
print("=" * 50)
print(".cat.categories — the unique category labels")
print("=" * 50)
# .cat.categories returns a pandas Index containing all unique category values
# These are the distinct labels that exist in the lookup table
categories = df_tips['gender'].cat.categories
print("Categories:", categories.tolist())  # → ['Female', 'Male']
print("Number of categories:", len(categories))

print()
print("=" * 50)
print(".cat.ordered — is there a defined ordering?")
print("=" * 50)
# .cat.ordered returns a boolean: True if categories have a defined order
# By default, categories created with astype('category') are UNORDERED
# Order matters for comparisons: 'Low' < 'Medium' < 'High'
is_ordered = df_tips['gender'].cat.ordered
print("Is ordered:", is_ordered)  # → False (gender has no natural order)

print()
print("=" * 50)
print(".cat.codes — the integer code for each row")
print("=" * 50)
# .cat.codes returns a Series of integer codes (int8 by default)
# Code 0 → 'Female', Code 1 → 'Male' (alphabetical by default)
# These codes are what pandas actually stores in memory
codes = df_tips['gender'].cat.codes
print("First 10 rows of codes:")
print(codes.head(10).to_string())
print(f"\nCode dtype: {codes.dtype}")  # int8 — only 1 byte per row!

print()
# Show the mapping between codes and labels
print("Code → Label mapping:")
for code, label in enumerate(df_tips['gender'].cat.categories):
    print(f"  Code {code} → '{label}'")

In [ ]:
print("=== Converting multiple columns to category at once ===")
# Identify which columns are candidates for category conversion
# Good candidates: columns with few unique values relative to total rows

# Check unique value counts per column
print("Unique values per column:")
for col in df_tips.columns:
    n_unique = df_tips[col].nunique()
    n_rows = len(df_tips)
    pct = n_unique / n_rows * 100
    flag = " ← good category candidate" if pct < 20 else ""
    print(f"  {col:15s}: {n_unique:3d} unique ({pct:5.1f}%){flag}")

print()
# Convert the clearly categorical columns
# List the columns that have a small, fixed set of meaningful labels
categorical_cols = ['smoker', 'day', 'time']

for col in categorical_cols:
    # Convert each column to category dtype
    df_tips[col] = df_tips[col].astype('category')
    print(f"  Converted '{col}': {df_tips[col].cat.categories.tolist()}")

print("\n=== Updated dtypes ===")
print(df_tips.dtypes)

---

## 5.2 Ordered Categories — Defining a Meaningful Order

Some categorical variables have a **natural order** that matters for comparisons and sorting:
- `risk_level`: Low < Medium < High
- `education`: High School < Bachelor's < Master's < PhD
- `size`: Small < Medium < Large < Extra Large

You can define this order using `pd.CategoricalDtype` with `ordered=True`.

In [ ]:
print("=== Creating an ORDERED categorical variable ===")

# Create a small DataFrame with a risk level column
df_risk = pd.DataFrame({
    'loan_id':    [101, 102, 103, 104, 105, 106],
    'amount':     [50000, 20000, 150000, 75000, 30000, 200000],
    'risk_level': ['High', 'Low', 'High', 'Medium', 'Low', 'Medium']
})

print("Before — risk_level as plain object:")
print(df_risk.to_string())
print(f"dtype: {df_risk['risk_level'].dtype}")

print()
# Define a CategoricalDtype with a specific ORDER
# categories=[...]: lists the valid values IN the desired order (Low first, High last)
# ordered=True: enables comparison operators (<, >, <=, >=) between categories
risk_order = pd.CategoricalDtype(
    categories=['Low', 'Medium', 'High'],  # Ordered from lowest to highest
    ordered=True                            # Enable ordering comparisons
)

# Convert the column using our ordered CategoricalDtype
df_risk['risk_level'] = df_risk['risk_level'].astype(risk_order)

print("After — risk_level as ordered category:")
print(df_risk['risk_level'])
print(f"\nCategories: {df_risk['risk_level'].cat.categories.tolist()}")
print(f"Ordered:    {df_risk['risk_level'].cat.ordered}")

print()
print("=== Ordering enables meaningful comparisons ===")
# Now < and > work correctly based on the defined category order
# This would NOT work on an unordered category
high_risk_mask = df_risk['risk_level'] > 'Low'
print("Loans with risk_level > 'Low':")
print(df_risk[high_risk_mask][['loan_id', 'amount', 'risk_level']].to_string())

print()
print("=== Sorting respects category order ===")
# .sort_values() on an ordered category sorts by the category order, not alphabetically
# Without ordering, 'High' < 'Low' < 'Medium' (alphabetical)
# With ordering, 'Low' < 'Medium' < 'High' (our defined order)
print(df_risk.sort_values('risk_level')[['loan_id', 'risk_level']].to_string())

---

## 5.3 Manipulating Categories — Add, Remove, Rename

After creating a categorical column, you can modify its categories without changing the data values.

In [ ]:
print("=== Starting point: gender as category ===")
# Confirm current state of categories
print(f"Categories: {df_tips['gender'].cat.categories.tolist()}")
print(f"Value counts:")
print(df_tips['gender'].value_counts().to_string())

print()
print("=" * 50)
print("ADD a new category label (no data yet — just the label)")
print("=" * 50)
# .cat.add_categories(): adds new valid labels to the category list
# No data is changed — it just expands the set of POSSIBLE values
# Useful before receiving data that will contain the new category
df_tips['gender'] = df_tips['gender'].cat.add_categories(['Non-binary', 'Prefer not to say'])
print(f"Categories after add: {df_tips['gender'].cat.categories.tolist()}")
print("(No actual data was changed — just expanded the valid label set)")

print()
print("=" * 50)
print("REMOVE a category (any matching rows become NaN)")
print("=" * 50)
# .cat.remove_categories(): removes the label(s) from the category list
# Any row that currently holds a removed label will become NaN
# Here we remove the labels we just added (no rows have those values, so no NaN created)
df_tips['gender'] = df_tips['gender'].cat.remove_categories(['Non-binary', 'Prefer not to say'])
print(f"Categories after remove: {df_tips['gender'].cat.categories.tolist()}")

print()
print("=" * 50)
print("RENAME category labels")
print("=" * 50)
# .cat.rename_categories(): renames the labels without changing the codes
# Pass a dictionary: {'old_name': 'new_name'}
# Or pass a list of new names in the same order as current categories
df_tips['gender'] = df_tips['gender'].cat.rename_categories({'Female': 'F', 'Male': 'M'})
print(f"Categories after rename: {df_tips['gender'].cat.categories.tolist()}")
print("First 5 values after rename:")
print(df_tips['gender'].head().to_string())

# Rename back to original labels for clean state
df_tips['gender'] = df_tips['gender'].cat.rename_categories({'F': 'Female', 'M': 'Male'})
print("\nRenamed back to original:", df_tips['gender'].cat.categories.tolist())

In [ ]:
print("=== PRACTICAL EXAMPLE: Category dtype in groupby analysis ===")
print()

# Group by the categorical 'day' column and calculate mean tip
# Category groupby includes ALL defined categories in the result
# even if some days have no data (avoids silently missing groups)
print("Mean tip by day:")
# .groupby(observed=False): include all categories even if some have no data
mean_tip_by_day = df_tips.groupby('day', observed=False)['tip'].mean()
print(mean_tip_by_day.to_string())

print()
print("Mean tip by gender:")
mean_tip_by_gender = df_tips.groupby('gender', observed=False)['tip'].mean()
print(mean_tip_by_gender.to_string())

print()
print("=== Full summary: counts and means by time and gender ===")
# Pivot table: see how tip amounts differ across time (Lunch/Dinner) and gender
summary = df_tips.pivot_table(
    values='tip',          # Column to aggregate
    index='time',          # Rows
    columns='gender',      # Columns
    aggfunc=['mean', 'count']  # Aggregation functions
)
print(summary.round(2).to_string())

---

# Summary

---

## Key Concepts at a Glance

### Part 2: Converting to Strings

| Concept | Code | Result dtype |
|---|---|---|
| Convert column to string | `df['col'].astype(str)` | `object` |
| Convert int to string for text ops | `df['ID'].astype(str).str.zfill(4)` | `object` |

### Part 3: Converting to Numerics

| Concept | Code | Notes |
|---|---|---|
| Clean conversion | `df['col'].astype(float)` | Raises error if any value is non-numeric |
| Messy data — crash | `pd.to_numeric(s, errors='raise')` | Default — strict |
| Messy data — coerce | `pd.to_numeric(s, errors='coerce')` | Bad values → NaN |
| Memory-efficient | `pd.to_numeric(s, downcast='integer')` | Smallest int type that fits |

### Part 4: Categorical Data

| Concept | Code | Benefit |
|---|---|---|
| Convert to category | `df['col'].astype('category')` | Less memory, faster groupby |
| Ordered category | `pd.CategoricalDtype(['Low','Med','High'], ordered=True)` | Enables `<`, `>`, meaningful sort |

### Part 5: Category `.cat` Accessor

| Attribute / Method | Code | Returns |
|---|---|---|
| Unique labels | `df['col'].cat.categories` | Index of labels |
| Integer codes | `df['col'].cat.codes` | int8 Series (0, 1, 2...) |
| Is ordered? | `df['col'].cat.ordered` | bool |
| Add labels | `df['col'].cat.add_categories(['New'])` | New category, no data change |
| Remove labels | `df['col'].cat.remove_categories(['Old'])` | Matching rows → NaN |
| Rename labels | `df['col'].cat.rename_categories({'A': 'B'})` | Labels renamed |

---

## When to Use `category` Dtype — Decision Guide

```
Should I convert this column to category?

├── Is the column text (object dtype)?  →  YES
│   ├── How many unique values?
│   │   ├── < 50% of total rows  →  ✅ Good candidate for category
│   │   └── > 50% of total rows  →  ❌ High cardinality — don't use category
│   │
│   └── Does it have a natural order (Low/Med/High)?
│       ├── YES  →  ✅ Use ordered=True CategoricalDtype
│       └── NO   →  ✅ Use astype('category') (unordered)
│
└── Is the column numeric?  →  Generally don't use category
    (Unless you're binning into ranges: '0-25k', '25-50k', '50k+')
```

---

## Self-Test Questions

1. What is the difference between `astype(str)` and `astype('category')`?
2. Why does a string column show `object` dtype even after `astype(str)`?
3. When would you use `pd.to_numeric(errors='coerce')` instead of `astype(float)`?
4. What does `.cat.codes` return? What integer type are the codes stored as?
5. Give two real-world examples of ordered categorical data and explain why order matters.
6. What happens to a row's value if you call `.cat.remove_categories()` on a category that the row currently holds?